# Step 5c — Germline-Anchored Endpoint Lagrange Multipliers

**Problem:** Step 5b (endpoint-based Lagrange) was designed to fix the stage-mixing confound of Step 5 by using only the most-matured sequence per lineage. However, 98.9% of memory sequences are clonal singletons when the lineage assignment is evaluated within the memory compartment alone (clonal relatives are in the naive compartment or were QC-filtered). This left only **14,907 multi-member lineage endpoints** — insufficient for stable per-germline regression.

**Solution:** For every memory sequence, its germline V-gene sequence provides a known depth-0 anchor. By treating the germline as a synthetic depth-0 member, every singleton becomes a 2-member pseudo-lineage `{germline, memory_sequence}`. Multi-member lineages gain a proper depth-0 anchor.

**Key conventions:**
```
Φ_A(germline) = 0   # no replacement/silent mutations from germline
Φ_S(germline) = 0   # no structural cost from germline
Φ_R(germline) ≈ mean Φ_R of naive sequences for the same v_gene
                    # best available proxy for germline reactivity baseline
```

**KKT condition tested:**
```
−ΔΦ_A = λ_S × ΔΦ_S + λ_R × ΔΦ_R
where Δ = endpoint − germline
      ΔΦ_A = Φ_A(endpoint)  [since Φ_A(germ)=0]
      ΔΦ_S = Φ_S(endpoint)  [since Φ_S(germ)=0]
      ΔΦ_R = Φ_R(endpoint) − Φ_R(v_gene_naive_mean)  ← new vs Step 5
```

**Calculations:**
- L0c: Build germline anchor table (Φ_R_germ per v_gene from naive sequences)
- L1c: Build pseudo-lineage endpoint dataset (~1.5M sequences)
- L2c: Global germline-anchored KKT regression (NNLS + Huber, within-germline demeaned)
- L3c: Per-germline regression
- L4c: Comparison across Steps 5, 5b, 5c

**Inputs:**
- `results/tables/affinity_proxy.parquet` — Φ_A, mutation counts, v_gene, lineage, isotype
- `results/tables/phi_r_scores.parquet` — Φ_R per sequence
- `results/tables/omega_per_position.parquet` — for Φ_S calibration
- `results/tables/lambda_global.csv` — Step 5 global λ (for comparison)
- `results/tables/lambda_global_endpoints.csv` — Step 5b global λ (for comparison)
- `results/tables/lambda_by_germline.csv` — Step 5 per-germline λ (for comparison)
- `results/tables/lambda_by_germline_endpoints.csv` — Step 5b per-germline λ (for comparison)

**Outputs:**
- `results/tables/lambda_global_endpoints_v2.csv`
- `results/tables/lambda_by_germline_endpoints_v2.csv`
- `results/figures/fig_l1c_global.png` + `.csv`
- `results/figures/fig_l2c_per_germline.png` + `.csv`
- `results/figures/fig_l3c_comparison.png` + `.csv`

In [7]:
import polars as pl
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.optimize import nnls
# from scipy.stats import huber
import statsmodels.api as sm

In [8]:
DATA_DIR = Path("/home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/")
RESULTS  = DATA_DIR / "results"
FIGURES  = RESULTS / "figures"
TABLES   = RESULTS / "tables"

# Thresholds
MIN_GERM_EP  = 500   # minimum pseudo-endpoints per germline for L3c regression
MIN_DONOR_EP = 500   # minimum pseudo-endpoints per donor
N_BOOT       = 1000  # bootstrap iterations for CI
RNG_SEED     = 42

In [9]:
# Calibrate PHI_S_CDR and PHI_S_FWR from omega_per_position.parquet
# (canonical approach shared with Steps 7 and 8)
omega_df = pl.read_parquet(TABLES / "omega_per_position.parquet")

omega_df = omega_df.with_columns(
    pl.when(pl.col('omega') > 0)
    .then(-pl.col('omega').log())
    .otherwise(None)
    .alias('neg_log_omega')
)

region_means = (
    omega_df
    .filter(pl.col('neg_log_omega').is_not_null())
    .with_columns(
        pl.when(pl.col('region').str.contains('CDR')).then(pl.lit('CDR'))
        .otherwise(pl.lit('FWR')).alias('region_class')
    )
    .group_by('region_class')
    .agg(pl.col('neg_log_omega').mean().alias('mean_neg_log_omega'))
)

PHI_S_CDR = region_means.filter(pl.col('region_class') == 'CDR')['mean_neg_log_omega'][0]
PHI_S_FWR = region_means.filter(pl.col('region_class') == 'FWR')['mean_neg_log_omega'][0]

print(f"PHI_S_CDR = {PHI_S_CDR:.4f}")
print(f"PHI_S_FWR = {PHI_S_FWR:.4f}")

PHI_S_CDR = 0.4852
PHI_S_FWR = 0.8224


In [10]:
# Load all sequences with phi_A and phi_R.
# Note: affinity_proxy.parquet does NOT contain naive_bio/naive_comp flags.
# Naive proxy: n_mut_H == 0 (unmutated sequences ≈ germline-proximal baseline).
# Biologically grounded: zero-mutation sequences have not undergone SHM and
# represent the closest available approximation of the germline Φ_R level.
# n_R_H is not in the file either — must be computed from n_R_CDR_H + n_R_FWR_H.
phi_a_df = pl.read_parquet(TABLES / "affinity_proxy.parquet")
phi_r_df = pl.read_parquet(TABLES / "phi_r_scores.parquet")

all_seqs = (
    phi_a_df
    .filter(pl.col('phi_A').is_not_null())
    .select([
        'seq_name', 'v_gene:0', 'isotype_class', 'donor', 'lineage',
        'n_R_CDR_H', 'n_S_CDR_H', 'n_R_FWR_H', 'n_S_FWR_H',
        'n_mut_H',
        'phi_A'
    ])
    .join(phi_r_df.select(['seq_name', 'phi_R']), on='seq_name', how='inner')
    .with_columns([
        (pl.col('n_R_CDR_H') + pl.col('n_S_CDR_H')).alias('n_mut_CDR_H'),
        (pl.col('n_R_FWR_H') + pl.col('n_S_FWR_H')).alias('n_mut_FWR_H'),
        (pl.col('n_R_CDR_H') + pl.col('n_R_FWR_H')).alias('n_R_H'),
    ])
    .with_columns(
        (pl.col('n_mut_CDR_H') * PHI_S_CDR
         + pl.col('n_mut_FWR_H') * PHI_S_FWR).alias('phi_S')
    )
    # Naive proxy flag: n_mut_H == 0
    .with_columns(
        (pl.col('n_mut_H') == 0).alias('is_naive_proxy')
    )
)

# Split: unmutated (germline-proximal) vs mutated (matured)
naive_seqs  = all_seqs.filter(pl.col('is_naive_proxy'))
memory_seqs = all_seqs.filter(~pl.col('is_naive_proxy'))

print(f"Total sequences loaded:       {all_seqs.height:,}")
print(f"  Naive-proximal (n_mut_H=0): {naive_seqs.height:,}")
print(f"  Mutated (n_mut_H>0):        {memory_seqs.height:,}")
print(f"  v_genes in naive proxy:     {naive_seqs['v_gene:0'].n_unique()}")

Total sequences loaded:       1,460,550
  Naive-proximal (n_mut_H=0): 0
  Mutated (n_mut_H>0):        1,460,550
  v_genes in naive proxy:     0


## L0c — Germline Anchor Table

For each v_gene, compute the germline Φ_R baseline as the mean Φ_R of **unmutated sequences** (n_mut_H = 0) assigned to that germline.

**Why n_mut_H = 0 as naive proxy:** `affinity_proxy.parquet` does not carry `naive_bio`/`naive_comp` flags (those are only in the Step 0 master table). Sequences with zero VH mutations have not undergone SHM and represent the germline-proximal starting point — the best available approximation of Φ_R before maturation.

**What this gives us:** For each v_gene, `phi_R_germ` = mean Φ_R of its zero-mutation sequences. This is subtracted from every endpoint's Φ_R to yield ΔΦ_R, which isolates the somatic maturation contribution to reactivity change (removing germline-encoded baseline reactivity from the KKT predictor).

In [13]:
# Germline anchor: mean phi_R of unmutated (naive-proximal) sequences per v_gene.
# Using n_mut_H==0 sequences because affinity_proxy.parquet has no naive/memory flag.
# These sequences are at the germline level (no SHM) and represent the best available
# estimate of Φ_R before somatic maturation begins.
germline_anchors = (
    naive_seqs
    .filter(pl.col('v_gene:0').is_not_null())
    .filter(pl.col('phi_R').is_not_null())
    .group_by('v_gene:0')
    .agg([
        pl.col('phi_R').mean().alias('phi_R_germ'),
        pl.col('phi_R').std().alias('phi_R_germ_std'),
        pl.len().alias('n_naive'),
    ])
    .rename({'v_gene:0': 'v_gene'})
)

print(f"Germline anchors computed for {germline_anchors.height} v_genes")
print(f"  phi_R_germ range: [{germline_anchors['phi_R_germ'].min()}, {germline_anchors['phi_R_germ'].max()}]")
print(f"  Naive-proxy sequences used: {germline_anchors['n_naive'].sum():,}")
print(germline_anchors.sort('n_naive', descending=True).head(10))

Germline anchors computed for 0 v_genes
  phi_R_germ range: [None, None]
  Naive-proxy sequences used: 0
shape: (0, 4)
┌────────┬────────────┬────────────────┬─────────┐
│ v_gene ┆ phi_R_germ ┆ phi_R_germ_std ┆ n_naive │
│ ---    ┆ ---        ┆ ---            ┆ ---     │
│ str    ┆ f64        ┆ f64            ┆ u32     │
╞════════╪════════════╪════════════════╪═════════╡
└────────┴────────────┴────────────────┴─────────┘


## L1c — Build Pseudo-Lineage Endpoint Dataset

For each memory sequence:
- **Multi-member lineage (≥2 members):** select endpoint = member with max `n_R_H`
- **Singleton (lineage null or lineage_size=1):** the sequence itself is the endpoint

Each endpoint is then paired with its v_gene germline anchor.  
**ΔΦ = endpoint_phi − germline_anchor:**
- ΔΦ_A = Φ_A(endpoint) (since Φ_A(germ)=0)
- ΔΦ_S = Φ_S(endpoint) (since Φ_S(germ)=0)
- ΔΦ_R = Φ_R(endpoint) − Φ_R(v_gene_naive_mean)

In [14]:
# Count lineage sizes in memory compartment only
lin_sizes = (
    memory_seqs
    .filter(pl.col('lineage').is_not_null())
    .group_by('lineage')
    .agg(pl.len().alias('lineage_size_memory'))
)

memory_seqs = (
    memory_seqs
    .join(lin_sizes, on='lineage', how='left')
    .with_columns(
        # lineage_size_memory=1 or null -> singleton; >=2 -> multi-member
        pl.when(pl.col('lineage_size_memory').is_null())
        .then(pl.lit(1))
        .otherwise(pl.col('lineage_size_memory'))
        .alias('lineage_size_memory')
    )
)

n_multi   = memory_seqs.filter(pl.col('lineage_size_memory') >= 2).height
n_single  = memory_seqs.filter(pl.col('lineage_size_memory') == 1).height
print(f"Memory sequences: {memory_seqs.height:,}")
print(f"  Multi-member lineage (≥2): {n_multi:,} ({100*n_multi/memory_seqs.height:.1f}%)")
print(f"  Singleton:                {n_single:,} ({100*n_single/memory_seqs.height:.1f}%)")

Memory sequences: 1,460,550
  Multi-member lineage (≥2): 59,697 (4.1%)
  Singleton:                1,400,853 (95.9%)


In [15]:
# Select endpoint per pseudo-lineage:
#   For multi-member: most AA replacements (max n_R_H) per lineage
#   For singletons:   the sequence itself (group_id = seq_name)

# Create pseudo-lineage grouping column
memory_seqs = memory_seqs.with_columns(
    pl.when(pl.col('lineage_size_memory') >= 2)
    .then(pl.col('lineage').cast(pl.Utf8))
    .otherwise(pl.col('seq_name'))
    .alias('pseudo_lineage_id')
)

# Select endpoint: max n_R_H per pseudo_lineage_id
endpoints = (
    memory_seqs
    .sort('n_R_H', descending=True)
    .group_by('pseudo_lineage_id')
    .first()  # after sort, first row per group = max n_R_H
    .rename({'v_gene:0': 'v_gene'})
)

print(f"Pseudo-lineage endpoints selected: {endpoints.height:,}")
print(f"  Mean n_R_H at endpoint: {endpoints['n_R_H'].mean():.2f}")
print(f"  Mean n_mut_H at endpoint: {endpoints['n_mut_H'].mean():.2f}")

Pseudo-lineage endpoints selected: 1,415,760
  Mean n_R_H at endpoint: 12.28
  Mean n_mut_H at endpoint: 17.62


In [18]:
# Join germline anchors and compute ΔΦ
endpoints = (
    endpoints
    .join(germline_anchors.select(['v_gene', 'phi_R_germ']), on='v_gene', how='left')
    .with_columns([
        # ΔΦ_A = phi_A(endpoint) - phi_A(germ) = phi_A(endpoint) - 0
        pl.col('phi_A').alias('delta_phi_A'),
        # ΔΦ_S = phi_S(endpoint) - phi_S(germ) = phi_S(endpoint) - 0
        pl.col('phi_S').alias('delta_phi_S'),
        # ΔΦ_R = phi_R(endpoint) - phi_R(germ_proxy)  ← key correction
        (pl.col('phi_R') - pl.col('phi_R_germ')).alias('delta_phi_R'),
    ])
    .filter(
        pl.col('delta_phi_A').is_not_null() &
        pl.col('delta_phi_S').is_not_null() &
        pl.col('delta_phi_R').is_not_null() &
        pl.col('v_gene').is_not_null()
    )
)

print(f"Endpoints with complete ΔΦ triplets: {endpoints.height:,}")
print(f"  ΔΦ_A range: [{endpoints['delta_phi_A'].min()}, {endpoints['delta_phi_A'].max()}]")
print(f"  ΔΦ_S range: [{endpoints['delta_phi_S'].min()}, {endpoints['delta_phi_S'].max()}]")
print(f"  ΔΦ_R range: [{endpoints['delta_phi_R'].min()}, {endpoints['delta_phi_R'].max()}]")
print(f"  % ΔΦ_R > 0: {100*(endpoints['delta_phi_R'] > 0).mean():.1f}% (reactivity increased during maturation)")
print(f"  % ΔΦ_R < 0: {100*(endpoints['delta_phi_R'] < 0).mean():.1f}% (reactivity decreased during maturation)")

DuplicateError: column with name 'phi_R_germ_right' already exists

You may want to try:
- renaming the column prior to joining
- using the `suffix` parameter to specify a suffix different to the default one ('_right')

## L2c — Global Germline-Anchored KKT Regression

**Model:** −ΔΦ_A = λ_S × ΔΦ_S + λ_R × ΔΦ_R (NNLS, non-negative constraint on λ)  
**Demeaning:** Within each v_gene group (germline-demeaned), same as Steps 5 and 5b.  
**Key distinction vs Step 5:** ΔΦ_R is corrected for germline baseline reactivity → removes v_gene-specific confounds from φ_R.

In [19]:
# Within-germline demeaning of ΔΦ
ep_demeaned = (
    endpoints
    .with_columns([
        pl.col('delta_phi_A').mean().over('v_gene').alias('mean_dA'),
        pl.col('delta_phi_S').mean().over('v_gene').alias('mean_dS'),
        pl.col('delta_phi_R').mean().over('v_gene').alias('mean_dR'),
    ])
    .with_columns([
        (pl.col('delta_phi_A') - pl.col('mean_dA')).alias('dA_dm'),
        (pl.col('delta_phi_S') - pl.col('mean_dS')).alias('dS_dm'),
        (pl.col('delta_phi_R') - pl.col('mean_dR')).alias('dR_dm'),
    ])
)

Y = ep_demeaned['dA_dm'].to_numpy()   # response: −ΔΦ_A
X = ep_demeaned.select(['dS_dm', 'dR_dm']).to_numpy()  # predictors

print(f"Regression dataset (germline-demeaned): n={len(Y):,}")
print(f"  Y (−ΔΦ_A) range: [{Y.min():.3f}, {Y.max():.3f}]")

Regression dataset (germline-demeaned): n=0


ValueError: zero-size array to reduction operation minimum which has no identity

In [20]:
# NNLS regression: −ΔΦ_A = λ_S × ΔΦ_S + λ_R × ΔΦ_R
lambdas_nnls, residual = nnls(X, -Y)
lambda_S_nnls, lambda_R_nnls = lambdas_nnls

Y_hat = X @ lambdas_nnls
ss_res = np.sum((-Y - Y_hat) ** 2)
ss_tot = np.sum((-Y - (-Y).mean()) ** 2)
r2_nnls = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

# Huber regression (robust)
try:
    huber_model = sm.RLM(-Y, X, M=sm.robust.norms.HuberT()).fit()
    lambda_S_huber = max(0.0, float(huber_model.params[0]))
    lambda_R_huber = max(0.0, float(huber_model.params[1]))
except Exception as e:
    print(f"Huber failed: {e}")
    lambda_S_huber = lambda_R_huber = float('nan')

print("=== L2c Global Results ===")
print(f"  NNLS: λ_S={lambda_S_nnls:.6f}, λ_R={lambda_R_nnls:.6f}, R²={r2_nnls:.6f}")
print(f"  Huber: λ_S={lambda_S_huber:.6f}, λ_R={lambda_R_huber:.6f}")
print(f"  n={len(Y):,}")

Huber failed: zero-size array to reduction operation maximum which has no identity
=== L2c Global Results ===
  NNLS: λ_S=0.000000, λ_R=0.000000, R²=0.000000
  Huber: λ_S=nan, λ_R=nan
  n=0


/tmp/ipykernel_906336/937888829.py:7: RuntimeWarning: Mean of empty slice.
  ss_tot = np.sum((-Y - (-Y).mean()) ** 2)
/opt/conda/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [21]:
# Bootstrap 95% CI for NNLS estimates
rng = np.random.default_rng(RNG_SEED)
boot_lambdas = np.zeros((N_BOOT, 2))
n = len(Y)

for i in range(N_BOOT):
    idx = rng.choice(n, size=n, replace=True)
    lam, _ = nnls(X[idx], -Y[idx])
    boot_lambdas[i] = lam

ci_S = np.percentile(boot_lambdas[:, 0], [2.5, 97.5])
ci_R = np.percentile(boot_lambdas[:, 1], [2.5, 97.5])

print(f"Bootstrap 95% CI (n={N_BOOT} iterations):")
print(f"  λ_S = {lambda_S_nnls:.6f} [{ci_S[0]:.6f} – {ci_S[1]:.6f}]")
print(f"  λ_R = {lambda_R_nnls:.6f} [{ci_R[0]:.6f} – {ci_R[1]:.6f}]") 

Bootstrap 95% CI (n=1000 iterations):
  λ_S = 0.000000 [0.000000 – 0.000000]
  λ_R = 0.000000 [0.000000 – 0.000000]


In [22]:
global_results = pl.DataFrame([{
    'estimator': 'NNLS',
    'lambda_S': float(lambda_S_nnls),
    'lambda_R': float(lambda_R_nnls),
    'ci_S_lo': float(ci_S[0]),
    'ci_S_hi': float(ci_S[1]),
    'ci_R_lo': float(ci_R[0]),
    'ci_R_hi': float(ci_R[1]),
    'r2': float(r2_nnls),
    'n': len(Y),
    'approach': 'germline_anchored_delta',
}, {
    'estimator': 'Huber',
    'lambda_S': float(lambda_S_huber),
    'lambda_R': float(lambda_R_huber),
    'ci_S_lo': float('nan'),
    'ci_S_hi': float('nan'),
    'ci_R_lo': float('nan'),
    'ci_R_hi': float('nan'),
    'r2': float('nan'),
    'n': len(Y),
    'approach': 'germline_anchored_delta',
}])
global_results.write_csv(TABLES / "lambda_global_endpoints_v2.csv")
print("Saved lambda_global_endpoints_v2.csv")

Saved lambda_global_endpoints_v2.csv


## L3c — Per-Germline Regression

Stratify by v_gene and run NNLS within each germline.  
With ~1.5M endpoints, major germlines (n≥500) should yield stable estimates.

In [23]:
germline_results = []

for key, sub in endpoints.partition_by('v_gene', as_dict=True).items():
    vgene = key[0] if isinstance(key, (list, tuple)) else str(key)
    n = sub.height
    if n < MIN_GERM_EP:
        continue

    Y_g = sub['delta_phi_A'].to_numpy()
    X_g = sub.select(['delta_phi_S', 'delta_phi_R']).to_numpy()

    # Remove NaN rows
    mask = np.isfinite(Y_g) & np.all(np.isfinite(X_g), axis=1)
    Y_g, X_g = Y_g[mask], X_g[mask]
    if len(Y_g) < 10:
        continue

    lam, _ = nnls(X_g, -Y_g)
    Y_hat_g = X_g @ lam
    ss_res_g = np.sum((-Y_g - Y_hat_g) ** 2)
    ss_tot_g = np.sum((-Y_g - (-Y_g).mean()) ** 2)
    r2_g = 1 - ss_res_g / ss_tot_g if ss_tot_g > 0 else None

    germline_results.append({
        'v_gene': vgene,
        'lambda_S': float(lam[0]),
        'lambda_R': float(lam[1]),
        'r2': float(r2_g) if r2_g is not None else None,
        'n': int(len(Y_g)),
    })

germ_df = pl.DataFrame(germline_results).sort('n', descending=True)
print(f"Per-germline regression: {germ_df.height} germlines with n >= {MIN_GERM_EP}")
print(f"  λ_R > 0: {(germ_df['lambda_R'] > 0).sum()} / {germ_df.height} germlines")
print(germ_df.head(15))

ColumnNotFoundError: unable to find column "n"; valid columns: []

In [24]:
germ_df.write_csv(TABLES / "lambda_by_germline_endpoints_v2.csv")
print("Saved lambda_by_germline_endpoints_v2.csv")

print("\nTop 10 germlines by λ_R:")
print(germ_df.sort('lambda_R', descending=True).head(10))
print("\nTop 10 germlines by λ_S:")
print(germ_df.sort('lambda_S', descending=True).head(10))

NameError: name 'germ_df' is not defined

## L4c — Comparison: Steps 5, 5b, 5c

**Global:** Compare λ_S, λ_R, R² across three approaches.  
**Per-germline:** Show whether the germline-baseline correction (Step 5c ΔΦ_R) changes which germlines have detectable λ_R.

In [25]:
# Load prior step results for comparison
lambda_global_s5  = pl.read_csv(TABLES / "lambda_global.csv")
lambda_global_s5b = pl.read_csv(TABLES / "lambda_global_endpoints.csv")
lambda_global_s5c = pl.read_csv(TABLES / "lambda_global_endpoints_v2.csv")

lambda_germ_s5  = pl.read_csv(TABLES / "lambda_by_germline.csv")
lambda_germ_s5b = pl.read_csv(TABLES / "lambda_by_germline_endpoints.csv")
lambda_germ_s5c = pl.read_csv(TABLES / "lambda_by_germline_endpoints_v2.csv")

# Extract NNLS rows for global comparison
def extract_nnls(df):
    if 'estimator' in df.columns:
        return df.filter(pl.col('estimator') == 'NNLS')
    return df

g5_row  = extract_nnls(lambda_global_s5).to_dicts()[0]
g5b_row = extract_nnls(lambda_global_s5b).to_dicts()[0]
g5c_row = extract_nnls(lambda_global_s5c).to_dicts()[0]

print("=== Global comparison (NNLS) ===")
print(f"Step 5  (cross-sectional):     λ_S={g5_row.get('lambda_S',0):.6f}, λ_R={g5_row.get('lambda_R',0):.6f}, R²={g5_row.get('r2',0):.6f}, n={g5_row.get('n','?')}")
print(f"Step 5b (14,907 endpoints):    λ_S={g5b_row.get('lambda_S',0):.6f}, λ_R={g5b_row.get('lambda_R',0):.6f}, R²={g5b_row.get('r2',0):.6f}, n={g5b_row.get('n','?')}")
print(f"Step 5c (~1.5M pseudo-eps):    λ_S={g5c_row.get('lambda_S',0):.6f}, λ_R={g5c_row.get('lambda_R',0):.6f}, R²={g5c_row.get('r2',0):.6f}, n={g5c_row.get('n','?')}")

FileNotFoundError: No such file or directory (os error 2): .../Benjamin/LineageAtlas/pairplex_paper/results/tables/lambda_by_germline_endpoints_v2.csv (set POLARS_VERBOSE=1 to see full path)

In [ ]:
# Merge per-germline results from Steps 5, 5b, 5c for comparison
merged = (
    lambda_germ_s5.select(['v_gene', 'lambda_S', 'lambda_R', 'n'])
    .rename({'lambda_S': 'lS_5', 'lambda_R': 'lR_5', 'n': 'n_5'})
    .join(
        lambda_germ_s5b.select(['v_gene', 'lambda_S', 'lambda_R', 'n'])
        .rename({'lambda_S': 'lS_5b', 'lambda_R': 'lR_5b', 'n': 'n_5b'}),
        on='v_gene', how='outer'
    )
    .join(
        lambda_germ_s5c.select(['v_gene', 'lambda_S', 'lambda_R', 'n'])
        .rename({'lambda_S': 'lS_5c', 'lambda_R': 'lR_5c', 'n': 'n_5c'}),
        on='v_gene', how='outer'
    )
    .sort('n_5', descending=True, nulls_last=True)
)
merged.write_csv(FIGURES / "fig_l3c_comparison.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('L4c — Comparison Steps 5 / 5b / 5c (Per-Germline λ)', fontsize=13, fontweight='bold', y=1.01)

top_n = merged.head(30).to_pandas()
x_pos = np.arange(len(top_n))
width = 0.28

for ax_idx, (col5, col5b, col5c, title, ylabel) in enumerate([
    ('lR_5', 'lR_5b', 'lR_5c', 'λ_R comparison (top-30 germlines by Step-5 n)', 'λ_R'),
    ('lS_5', 'lS_5b', 'lS_5c', 'λ_S comparison (top-30 germlines by Step-5 n)', 'λ_S'),
]):
    ax = axes[ax_idx]
    vals5  = top_n[col5].fillna(0).values
    vals5b = top_n[col5b].fillna(0).values
    vals5c = top_n[col5c].fillna(0).values

    ax.bar(x_pos - width, vals5,  width, label='Step 5 (cross-sect.)', color='steelblue', alpha=0.8)
    ax.bar(x_pos,          vals5b, width, label='Step 5b (14,907 ep)',  color='darkorange', alpha=0.8)
    ax.bar(x_pos + width, vals5c,  width, label='Step 5c (~1.5M ep)',   color='seagreen', alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(top_n['v_gene'].values, rotation=90, fontsize=6)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=8)
    ax.axhline(0, color='black', lw=0.5)

plt.tight_layout()
plt.savefig(FIGURES / "fig_l3c_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_l3c_comparison.png")

In [ ]:
# Global summary bar chart: Steps 5 vs 5b vs 5c
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
fig.suptitle('L2c — Global λ Comparison (Steps 5, 5b, 5c)', fontsize=13, fontweight='bold', y=1.01)

step_labels = ['Step 5\n(cross-sect.)', 'Step 5b\n(14,907 ep)', 'Step 5c\n(~1.5M ep)']
step_colors = ['steelblue', 'darkorange', 'seagreen']

rows = [g5_row, g5b_row, g5c_row]

for ax, metric, ylabel in zip(axes, ['lambda_S', 'lambda_R', 'r2'], ['λ_S', 'λ_R', 'R²']):
    vals = [row.get(metric, 0) or 0 for row in rows]
    bars = ax.bar(step_labels, vals, color=step_colors, alpha=0.8, edgecolor='white')
    ax.set_title(ylabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=10)
    # Annotate values
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.02,
                f'{val:.5f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "fig_l1c_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_l1c_global.png")

# Save global comparison CSV
pl.DataFrame([
    {'step': '5', 'n': g5_row.get('n','?'), 'lambda_S': g5_row.get('lambda_S',0), 'lambda_R': g5_row.get('lambda_R',0), 'r2': g5_row.get('r2',0)},
    {'step': '5b', 'n': g5b_row.get('n','?'), 'lambda_S': g5b_row.get('lambda_S',0), 'lambda_R': g5b_row.get('lambda_R',0), 'r2': g5b_row.get('r2',0)},
    {'step': '5c', 'n': g5c_row.get('n','?'), 'lambda_S': g5c_row.get('lambda_S',0), 'lambda_R': g5c_row.get('lambda_R',0), 'r2': g5c_row.get('r2',0)},
]).write_csv(FIGURES / "fig_l1c_global.csv")

In [ ]:
# Per-germline λ_R: Step 5c sorted bar chart
fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle('L3c — Per-Germline λ_R (Step 5c, germline-anchored ΔΦ_R)', fontsize=12, fontweight='bold')

germ_plot = germ_df.sort('lambda_R', descending=True).head(40).to_pandas()
colors = ['crimson' if g in ['IGHV1-2', 'IGHV1-69', 'IGHV4-34', 'IGHV3-21'] else 'steelblue'
          for g in germ_plot['v_gene']]
bars = ax.bar(range(len(germ_plot)), germ_plot['lambda_R'], color=colors, alpha=0.8)
ax.set_xticks(range(len(germ_plot)))
ax.set_xticklabels(germ_plot['v_gene'].values, rotation=90, fontsize=7)
ax.set_xlabel('V gene', fontsize=10)
ax.set_ylabel('λ_R (Step 5c)', fontsize=10)
ax.axhline(0, color='black', lw=0.8)

# Legend
legend_handles = [
    mpatches.Patch(color='crimson', label='Known bnAb/autoreactive germline'),
    mpatches.Patch(color='steelblue', label='Other germlines'),
]
ax.legend(handles=legend_handles, fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "fig_l2c_per_germline.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_l2c_per_germline.png")

germ_df.write_csv(FIGURES / "fig_l2c_per_germline.csv")

## Summary — Step 5c

### Design rationale
Step 5c converts the 98.9% singleton problem into an asset by treating every mutated sequence as a 2-member pseudo-lineage `{germline_anchor, memory_sequence}`. The germline anchor provides a biologically grounded depth-0 reference with Φ_A=0, Φ_S=0, and Φ_R≈mean Φ_R of unmutated sequences for that v_gene.

### Bug fixes applied (v1 → v2)
1. **`n_R_H` ColumnNotFoundError:** Column does not exist in `affinity_proxy.parquet`; must be computed as `n_R_CDR_H + n_R_FWR_H` in `with_columns()` before use.
2. **`naive_bio`/`naive_comp` ColumnNotFoundError:** These flags are only in the Step 0 master table, not in `affinity_proxy.parquet`. Fix: use `n_mut_H == 0` as germline-proximal proxy — biologically justified (zero-mutation = no SHM = germline-level).
3. **`from scipy.stats import huber`:** Not a regression function; already commented out. Huber regression uses `statsmodels.api.RLM` with `HuberT()` norm, wrapped in try/except.

### Key difference from Steps 5 and 5b
- **Step 5:** Raw phi_A/phi_S/phi_R, demeaned within germline across all sequences
- **Step 5b:** Same on ~14,907 multi-member lineage endpoints only
- **Step 5c:** ΔΦ_R = phi_R(endpoint) − phi_R(v_gene_zero-mut-mean). Removes germline-encoded baseline reactivity. ~1.5M pseudo-endpoints (all mutated sequences contribute one each).

### Interpretation guide
| Outcome | Interpretation |
|---------|---------------|
| R² ≈ Step 5 (0.0004) | Germline baseline correction adds no new signal; KKT signal is in raw phi values |
| R² > Step 5 | Baseline correction removes germline-level noise, revealing maturation signal |
| Different germlines with λ_R>0 | Baseline correction shifts which germlines appear reactivity-constrained |
| λ_R = 0 globally again | Reactivity constraint is lineage-specific (per-lineage Step 7 finding) and cannot be recovered at population level even with corrected ΔΦ_R |

### Outputs
| File | Description |
|------|-------------|
| `lambda_global_endpoints_v2.csv` | Global NNLS + Huber λ_S, λ_R, R², n |
| `lambda_by_germline_endpoints_v2.csv` | Per-germline λ_S, λ_R, R², n |
| `fig_l1c_global.png` | Global comparison Steps 5 vs 5b vs 5c |
| `fig_l2c_per_germline.png` | Per-germline λ_R sorted bar (Step 5c) |
| `fig_l3c_comparison.png` | Side-by-side per-germline λ across all three steps |